# Day 10 — XGBoost: native missing-value handling and an honest CV-vs-test lesson

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

## Setup — a new preprocessing pipeline, on purpose

Day 6–9 all shared one feature set: `pclass`, `fare`, `who` (sex+age folded into a 3-way categorical), `family_size` — built by median-imputing `age` *before* deriving `who`, then dropping `age`/`embarked`/`sex` entirely. By the time that pipeline reaches any model, there are zero missing values left anywhere — `who`'s imputation absorbs `age`'s missingness. That's fine for Day 6–9's purposes, but it makes today's actual subject — how XGBoost handles missing values natively — impossible to demonstrate. There's nothing left to hand it.

So this notebook only, `age` stays raw with its ~20% NaNs intact, and categoricals (`sex`, `pclass`, `embarked`) go through a real `ColumnTransformer` + `OneHotEncoder` instead of manual integer mapping. Same `train_test_split(test_size=0.2, random_state=42)` as every prior day, so the 712/179 split is unchanged — but the feature *representation* is different from Day 6–9. Keep that in mind for the leaderboard in Step 4: accuracy deltas there partly reflect the richer feature set, not only the algorithm.

In [ ]:
df = sns.load_dataset("titanic")
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)

numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "pclass", "embarked"]

print("train shape:", x_train.shape, " test shape:", x_test.shape)
print(
    "age NaNs in train:", x_train["age"].isna().sum(),
    " embarked NaNs in train:", x_train["embarked"].isna().sum(),
)

## Step 1 — what XGBoost adds over Day 9's gradient boosting

Day 9 built gradient boosting from scratch: fit a shallow regression tree to the residual `y - p`, scale it by a learning rate, add it to a running log-odds score, repeat. XGBoost is the same idea — sequential trees correcting the ensemble's remaining error — with three concrete upgrades on top of that mechanism:

**Second-order splits.** Day 9's trees were fit with plain squared-error regression on the residual — a first-order (gradient-only) approximation of the loss. XGBoost's split-finding uses both the gradient *and* the Hessian (second derivative) of the loss at each point, a Newton step rather than a gradient step. That lets it size each leaf's contribution more precisely instead of relying purely on the learning rate to keep updates sane.

**Explicit regularization.** Every tree adds `gamma` (a minimum-gain threshold before a split is worth making) and L1/L2 penalties on leaf weights directly into the objective it optimizes at each split — not a separate step bolted on afterward, but baked into what "best split" means. Day 9's from-scratch trees had no such penalty; the only thing holding them back from overfitting was `max_depth` and the learning rate.

**Native missing-value handling.** Day 9's residual-fitting trees needed complete, numeric input — hence the median-imputed `age` used since Day 6. XGBoost's split-finding instead *learns a default direction* for missing values at every split, from the training data itself, rather than requiring missingness to be resolved upstream. That's today's Step 2.

First, get it running and sanity-check it against a model already trusted from Day 9: same `n_estimators=100, learning_rate=0.1, max_depth=2` sklearn `GradientBoostingClassifier`, same imputed features, both fit once.

In [ ]:
pre_imputed = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", OneHotEncoder(drop="if_binary"), categorical_features),
])

xgb_baseline = Pipeline([
    ("pre", pre_imputed),
    ("clf", XGBClassifier(random_state=42, eval_metric="logloss")),
])
xgb_baseline.fit(x_train, y_train)
xgb_test_acc = accuracy_score(y_test, xgb_baseline.predict(x_test))

gb_baseline = Pipeline([
    ("pre", pre_imputed),
    ("clf", GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42
    )),
])
gb_baseline.fit(x_train, y_train)
gb_test_acc = accuracy_score(y_test, gb_baseline.predict(x_test))

print(f"XGBClassifier (default params, imputed features):           test acc={xgb_test_acc:.4f}")
print(f"sklearn GradientBoostingClassifier (n=100, lr=0.1, depth=2): test acc={gb_test_acc:.4f}")

## Step 2 — native missing-value handling, tested directly

Two pipelines, identical except for how `age`'s missing values are treated: `pre_imputed` fills them with the median before XGBoost ever sees them (what every prior day has done); `pre_native` passes `age` straight through with its NaNs intact, letting XGBoost's own split-finding decide, per split, which direction a missing value should default to. Both compared on identical `XGBClassifier(random_state=42)` settings, test accuracy and 5-fold train-only CV.

In [ ]:
pre_native = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(drop="if_binary"), categorical_features),
])

for name, pre in [("imputed", pre_imputed), ("native-missing", pre_native)]:
    pipe = Pipeline([
        ("pre", pre),
        ("clf", XGBClassifier(random_state=42, eval_metric="logloss")),
    ])
    pipe.fit(x_train, y_train)
    test_acc = accuracy_score(y_test, pipe.predict(x_test))
    cv = cross_val_score(pipe, x_train, y_train, cv=5)
    print(f"{name:<15}: test acc={test_acc:.4f}, CV acc={cv.mean():.4f} (+/-{cv.std():.4f})")

Real numbers, not scripted ones: `imputed` gets test acc=0.7989, CV acc=0.7992 (+/-0.0176); `native-missing` gets test acc=0.8101, CV acc=0.7978 (+/-0.0142). CV is essentially a wash — 0.7992 vs 0.7978 is a 0.0014 gap, well inside either config's own +/- std, not a real difference. Test accuracy nudges in native's favor, +1.12 points (0.8101 vs 0.7989), even though native's CV *mean* was marginally the lower of the two — a small, concrete reminder that a CV mean and one 179-row test draw aren't required to move in the same direction.

Worth being honest about what this isn't: not a dramatic case for native handling. `age` is ~20% missing (140 of 712 training rows) concentrated in one feature, and nothing here tests whether *being* missing is itself predictive — it's just ordinary missingness, the kind median imputation handles reasonably well. XGBoost's learned split direction for the missing branch converges to something close enough to "impute with a typical value" that the two pipelines land in the same neighborhood. Native handling would likely separate more clearly from imputation on heavier missingness, or when missingness itself carries signal (a sensor that only fails under specific conditions, a form field skipped only by one demographic) — neither applies to `age` on the Titanic.

## Step 3 — CV-tune it properly

Grid search over `n_estimators × max_depth × learning_rate` (3×3×3 = 27 combinations), `cross_val_score(cv=5)` on `x_train` only — same train-only CV discipline as every grid search since Day 7. Sort by CV mean, print the top 5, refit the single best config on the full training set, and touch `x_test` exactly once for a final number.

In [ ]:
n_estimators_grid = [50, 100, 200]
max_depth_grid = [2, 3, 4]
lr_grid = [0.05, 0.1, 0.2]

results = []
for n_est in n_estimators_grid:
    for depth in max_depth_grid:
        for lr in lr_grid:
            pipe = Pipeline([
                ("pre", pre_native),
                ("clf", XGBClassifier(
                    n_estimators=n_est, max_depth=depth, learning_rate=lr,
                    random_state=42, eval_metric="logloss",
                )),
            ])
            scores = cross_val_score(pipe, x_train, y_train, cv=5)
            results.append((n_est, depth, lr, scores.mean(), scores.std()))

results.sort(key=lambda r: -r[3])
print("top 5 configs by CV mean:")
for n_est, depth, lr, mean, std in results[:5]:
    print(
        f"  n_estimators={n_est:>3}, max_depth={depth}, lr={lr:<4}: "
        f"CV acc={mean:.4f} (+/-{std:.4f})"
    )

best_n, best_depth, best_lr, best_cv, best_std = results[0]
print(
    f"\nBest: n_estimators={best_n}, max_depth={best_depth}, lr={best_lr}, "
    f"CV acc={best_cv:.4f} (+/-{best_std:.4f})"
)

best_xgb = Pipeline([
    ("pre", pre_native),
    ("clf", XGBClassifier(
        n_estimators=best_n, max_depth=best_depth, learning_rate=best_lr,
        random_state=42, eval_metric="logloss",
    )),
])
best_xgb.fit(x_train, y_train)
test_preds = best_xgb.predict(x_test)
final_test_acc = accuracy_score(y_test, test_preds)
cm = confusion_matrix(y_test, test_preds)

print(f"\nFinal ONE-TIME test accuracy: {final_test_acc:.4f}")
print("confusion matrix:\n", cm)

Grid search over 27 configs lands its best CV config at `n_estimators=100, max_depth=3, lr=0.1`, CV acc=0.8329 (+/-0.0055) — a notably tight std compared to most of this week's grid searches (Day 8's best was +/-0.0218, Day 9's +/-0.0152). Final ONE-TIME test accuracy: 0.8156.

That's a real CV-to-test gap worth naming plainly, even though it's more modest than the dramatic version described in the pasted lesson (their CV winner was the week's *highest* CV and *worst* test result; ours is respectable on both — third-highest CV of the week, and clearly ahead of Day 6's plain logistic regression on test). Two honest reasons for the gap that don't require anything to have gone wrong:

1. **179 test rows is a small, genuinely noisy final check** — the same caveat that's applied to every one-time test number since Day 7.
2. **Searching 27 configurations and keeping the single best CV score is itself a mild form of overfitting** — not to the test set (never touched until the final line), but to the specific 5-way split of these particular 712 training rows. The winning config is partly there because it happened to fit *this* CV partition well, not purely because it's the best config in general. A subtler cousin of the classic test-set-peeking trap — the optimism just shows up one level removed, in the CV estimate itself.

## Step 4 — the complete Week leaderboard, honestly read

Day 6–9's numbers below are the real, already-committed results from those notebooks (`who`/`pclass`/`fare`/`family_size` feature set). Day 10's row is what this notebook actually just computed, on the richer one-hot feature set described in Setup — not a straight apples-to-apples comparison, and the read below says so rather than pretending otherwise.

In [ ]:
print(f"{'Model':<38}{'CV accuracy':>14}{'Test accuracy':>16}")
print(f"{'Day 6 logistic regression (L2)':<38}{'—':>14}{0.8101:>16.4f}")
print(f"{'Day 7 single tree (CV-tuned)':<38}{0.8286:>14.4f}{0.8212:>16.4f}")
print(f"{'Day 8 random forest (CV-tuned)':<38}{0.8370:>14.4f}{0.8212:>16.4f}")
print(f"{'Day 9 gradient boosting (CV-tuned)':<38}{0.8384:>14.4f}{0.8212:>16.4f}")
print(f"{'Day 10 XGBoost (CV-tuned)':<38}{best_cv:>14.4f}{final_test_acc:>16.4f}")

Every tree-based model this week — Day 7 through Day 10 — clusters in a tight band: test accuracy from 0.8156 to 0.8212, none decisively ahead of the others on 179 held-out rows. Day 10's CV (0.8329) sits comfortably in the middle of the pack, below Day 8 and Day 9's CV but above Day 7's — and its test accuracy (0.8156) is the lowest of the four tree ensembles, though still ahead of Day 6's plain logistic regression.

The stated caveat from Setup matters here: Day 10 used a different, richer feature representation (one-hot `sex`/`pclass`/`embarked` plus raw `age`/`sibsp`/`parch`/`fare`) than Day 6–9's compact `who`/`pclass`/`fare`/`family_size` set, so this table isn't a clean single-variable comparison of "which algorithm is best" — the feature engineering changed too. What *is* comparable: XGBoost, tuned, landed in the same rough neighborhood as random forest and from-scratch gradient boosting on this dataset — consistent with the honest read Day 8 already established, that 891 rows is too small to crown an algorithmic winner with any real confidence.

## Step 5 — feature importances

`best_xgb`'s importances, compared against Day 8's random forest and Day 9's gradient boosting — both computed on the `who`/`pclass`/`fare`/`family_size` feature set, so this is a qualitative comparison (which underlying signal each method leans on) rather than a literal per-column match. `who` folded sex and age together into one categorical; here they're separate columns, so watch for whether `sex_male` alone recovers a similar share of the credit `who` got before.

In [ ]:
feature_names = best_xgb.named_steps["pre"].get_feature_names_out()
importances = best_xgb.named_steps["clf"].feature_importances_

print("XGBoost feature importances:")
for name, imp in sorted(zip(feature_names, importances), key=lambda t: -t[1]):
    print(f"  {name:<20}: {imp:.4f}")

print("\nDay 8 random forest (who/pclass/fare/family_size feature set) importances, for comparison:")
day8 = {"who": 0.4244, "fare": 0.3382, "pclass": 0.1199, "family_size": 0.1175}
for name, imp in day8.items():
    print(f"  {name:<20}: {imp:.4f}")

print("\nDay 9 gradient boosting (same feature set as Day 8) importances, for comparison:")
day9 = {"who": 0.5333, "fare": 0.1884, "pclass": 0.1700, "family_size": 0.1083}
for name, imp in day9.items():
    print(f"  {name:<20}: {imp:.4f}")

Real importances: `sex_male` alone takes 0.4890, `pclass_3` + `pclass_1` together add 0.2882 — over three-quarters of total importance sitting on just sex and class, mirroring how heavily Day 8 and Day 9's `who` (which folds sex and age together) dominated their own rankings (0.4244 and 0.5333 respectively). `fare` is a clear outlier across methods: sixth place here at 0.0379, versus second place at 0.3382 in Day 8's random forest — the same raw column, wildly different credit assigned depending on the method and what other features are competing for splits. `age`, now present as its own raw column instead of being folded into `who`, gets very little direct credit (0.0322) despite being the feature this notebook spent two steps on.

The real lesson, grounded in this notebook's own numbers rather than assumed in advance: feature importance is method-relative, not a fixed property of the data. Three tree-based methods this week, three visibly different importance pictures, on people who are — feature-set differences aside — largely the same 712 training rows. "Which feature matters" depends on how the model is built, not just what's in the dataset.

## Step 6 — exact per-prediction attribution (TreeSHAP), not just dataset-wide importance

Everything in Step 5 was a dataset-wide, unsigned, method-specific score — it can't answer "why did the model predict *this particular person* survived?" For that, XGBoost can compute **exact Shapley values per row** via `Booster.predict(..., pred_contribs=True)` — no new dependency, it's built into the library already installed. For every row, it returns one signed contribution per feature (in log-odds/margin units, same scale as `output_margin=True`) plus a bias term, and — unlike gain, weight, or cover — these are **exactly additive**: they sum to that row's predicted margin, not just "roughly explain" it.

This is the real fix to last message's question: a coefficient-like number, per feature, per prediction, with a sign and a consistent unit — the closest tree ensembles get to what Day 6's logistic regression weights gave for free.

In [ ]:
import xgboost as xgb

X_test_native = best_xgb.named_steps["pre"].transform(x_test)
dtest = xgb.DMatrix(X_test_native, feature_names=list(feature_names))
booster = best_xgb.named_steps["clf"].get_booster()

contribs = booster.predict(dtest, pred_contribs=True)  # (n_rows, n_features + 1); last col = bias
margins = booster.predict(dtest, output_margin=True)

print("max |sum(contribs) - predicted margin| across all test rows:",
      float(np.abs(contribs.sum(axis=1) - margins).max()))

In [ ]:
row = 3
print(f"Row {row}: true label={y_test.values[row]}, predicted margin={margins[row]:.4f}")
print()
for name, val in sorted(
    zip(list(feature_names) + ["bias"], contribs[row]), key=lambda t: -abs(t[1])
):
    print(f"  {name:<20}: {val:+.4f}")

Row 3 (original passenger index 720): a 6-year-old female, 2nd class, embarked Southampton, fare 33.0, no siblings/spouse aboard, 1 parent/child aboard. True label: survived. Predicted margin: +2.6893 (sigmoid → ~0.94 predicted probability of survival) — confident and correct. Verified across all 179 test rows: `sum(contribs) == predicted margin` to within 1.7e-6, exactly the additivity property gain/weight/cover don't have.

One easy misread worth naming directly: `cat__sex_male` shows **+1.0100** and `cat__pclass_3` shows **+1.2361** even though *this row's actual value for both is 0* (she's female, and not 3rd class). That's not a bug — for a one-hot column, `0` is exactly as informative as `1`. Knowing `sex_male = 0` tells the model "this passenger is female," which is itself a huge, well-known survival signal — its SHAP contribution isn't small just because the encoded number is small. Same for `pclass_3 = 0`: knowing someone is *not* in 3rd class is informative on its own. Don't read a one-hot SHAP value's size as proportional to the stored 0/1 — read it as "how much did knowing this fact move the prediction."

One more real detail, worth connecting back to Step 5: `cat__pclass_2` contributes exactly **+0.0000** here, even though this passenger *is* in 2nd class. With all three `pclass` dummies kept (no `drop='first'` for the 3-category column), `pclass_2` is redundant once `pclass_1` and `pclass_3` are both known to be 0 — the trees never needed to split on it directly, so it earns zero attribution. Same reason it showed 0.0000 importance in Step 5's aggregate ranking — this isn't a new fact, just the row-level version of it.

In [ ]:
# Aggregate: mean |contribution| across all test rows — a signed-attribution-based importance,
# directly comparable to (but computed completely differently from) Step 5's gain ranking.
mean_abs_contrib = np.abs(contribs[:, :-1]).mean(axis=0)
mean_abs_contrib = mean_abs_contrib / mean_abs_contrib.sum()

print("Mean |SHAP contribution| ranking (normalized):")
for name, val in sorted(zip(feature_names, mean_abs_contrib), key=lambda t: -t[1]):
    print(f"  {name:<20}: {val:.4f}")

print("\nStep 5's gain-based ranking, for comparison:")
for name, imp in sorted(zip(feature_names, importances), key=lambda t: -t[1]):
    print(f"  {name:<20}: {imp:.4f}")

The top two features agree between methods — `sex_male` and `pclass_3` lead both rankings, and at similar relative weight (0.436/0.489 and 0.188/0.217). Below that, they diverge meaningfully: gain ranks `age` 8th (0.0322) and `fare` 6th (0.0379); mean |SHAP contribution| ranks them 3rd (0.0991) and 4th (0.0973) — nearly tied for third place, well ahead of `pclass_1` and `sibsp`, which gain had ranked above both.

This is the concrete version of the point from earlier in this conversation: even holding the model completely fixed — same `best_xgb`, same fit, same trees — *which importance method you ask* changes the ranking. Gain measures average loss-reduction per split a feature is used in; mean |SHAP contribution| measures how much each feature actually moved individual predictions, averaged over every test row. `age` can be used in relatively few, low-gain splits (so it ranks low on gain) while still shifting many individual predictions by a moderate, consistent amount (so it ranks higher on mean |SHAP|) — two honest, different answers to "how much did this feature matter," from the same model.